In [1]:
# ==============================================================================
# CELL 0: OPUS BOOTSTRAP (SETUP, CONNECTIVITY & AESTHETICS)
# ==============================================================================
import warnings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import geopandas as gpd
from shapely.geometry import Point
import hdbscan
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# --- 1. HYGIENE & SOP 0405 ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

DUMP_DIR = '/workspaces/pienza/data/dumped_files'
os.makedirs(DUMP_DIR, exist_ok=True)
PREFIX = "0405"

# --- 2. CONNECTIVITY ---
DB_PATH = '/workspaces/pienza/data/pienza.db'
db_engine = create_engine(f'sqlite:///{DB_PATH}')

# --- 3. VISUAL CANON ---
OPUS_PURPLE = '#440154'; OPUS_TEAL = '#21918c'; OPUS_BG = '#FAFAFA'
plt.rcParams.update({'figure.facecolor': OPUS_BG, 'axes.facecolor': OPUS_BG})

print(f"✅ Production Map Engine (SOP {PREFIX}) Live.")

✅ Production Map Engine (SOP 0405) Live.


In [9]:
# ==============================================================================
# CELL: LA PROBADITA DE HDBSCAN (VERSIÓN BLINDADA Y AUTOCONTENIDA)
# ==============================================================================
# !pip install hdbscan # Ya está instalado, no es necesario de nuevo

import hdbscan
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🤖 **Invocando a la Bestia: HDBSCAN**"))

# --- 1. RECONSTRUCCIÓN DE DATOS (EL BLINDAJE) ---
# Nos aseguramos de que estamos usando el dataset completo y correcto
query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)

coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
print(f"✅ Datos reconstruidos para HDBSCAN. Shape: {coords_scaled.shape}")

# 2. INICIALIZAR Y AJUSTAR
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=25,
    min_samples=5,
    gen_min_span_tree=True,
    cluster_selection_epsilon=0.0
)

print("⏳ HDBSCAN está pensando...")
cluster_labels = clusterer.fit_predict(coords_scaled)

# AHORA las longitudes coinciden
df_hdbscan_analysis['hdbscan_cluster_id'] = cluster_labels
print("✅ Clustering HDBSCAN completado.")

# 3. ANÁLISIS
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados de HDBSCAN:**\n*   **Clusters Estables Encontrados:** `{n_clusters}`\n*   **Puntos de Ruido:** `{n_noise}`"))

# 4. VISUALIZACIÓN
fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="dropoff_lat",
    lon="dropoff_lon",
    color="hdbscan_cluster_id",
    mapbox_style="carto-positron",
    zoom=10.5, height=800,
    title='Mapa de Clusters HDBSCAN (min_cluster_size=25)',
    color_continuous_scale=px.colors.qualitative.Plotly
)
fig.show()

### 🤖 **Invocando a la Bestia: HDBSCAN**

✅ Datos reconstruidos para HDBSCAN. Shape: (4760, 2)
⏳ HDBSCAN está pensando...
✅ Clustering HDBSCAN completado.


### 🗺️ **Resultados de HDBSCAN:**
*   **Clusters Estables Encontrados:** `45`
*   **Puntos de Ruido:** `2144`

/tmp/ipykernel_8072/1484902954.py:43: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [10]:
# ==============================================================================
# CELL: HDBSCAN CON DOCTRINA "JITTER" (VISUALIZACIÓN MEJORADA)
# ==============================================================================
# !pip install hdbscan # Ya está instalado

import hdbscan
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**"))

# --- 1. RECONSTRUCCIÓN DE DATOS ---
query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)

coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
print(f"✅ Datos reconstruidos para HDBSCAN. Shape: {coords_scaled.shape}")

# --- 2. EJECUCIÓN DE HDBSCAN ---
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5)
print("⏳ HDBSCAN está pensando...")
cluster_labels = clusterer.fit_predict(coords_scaled)
df_hdbscan_analysis['hdbscan_cluster_id'] = cluster_labels
print("✅ Clustering HDBSCAN completado.")

# --- 3. ANÁLISIS DE RESULTADOS ---
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados de HDBSCAN:**\n*   **Clusters Estables Encontrados:** `{n_clusters}`\n*   **Puntos de Ruido:** `{n_noise}`"))

# --- 4. IMPLEMENTACIÓN DE LA DOCTRINA "JITTER" ---
np.random.seed(42)
noise_level = 0.0008 # Nivel de ruido sutil, ~100m. Ajusta si es necesario.

# Creamos nuevas columnas para la visualización
df_hdbscan_analysis['lat_viz'] = df_hdbscan_analysis['dropoff_lat'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
df_hdbscan_analysis['lon_viz'] = df_hdbscan_analysis['dropoff_lon'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
print(f"✅ Doctrina 'Jitter' aplicada con un nivel de ruido de {noise_level}.")

# --- 5. VISUALIZACIÓN MEJORADA ---
fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz", # <--- USAMOS LAS NUEVAS COLUMNAS
    lon="lon_viz", # <--- USAMOS LAS NUEVAS COLUMNAS
    color="hdbscan_cluster_id",
    mapbox_style="carto-positron",
    zoom=10.5, height=800,
    title='Mapa de Clusters HDBSCAN con Jitter (min_cluster_size=25)',
    color_continuous_scale=px.colors.qualitative.Plotly,
    category_orders={'hdbscan_cluster_id': sorted(df_hdbscan_analysis['hdbscan_cluster_id'].unique())}
)
fig.show()

### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**

✅ Datos reconstruidos para HDBSCAN. Shape: (4760, 2)
⏳ HDBSCAN está pensando...
✅ Clustering HDBSCAN completado.


### 🗺️ **Resultados de HDBSCAN:**
*   **Clusters Estables Encontrados:** `45`
*   **Puntos de Ruido:** `2144`

✅ Doctrina 'Jitter' aplicada con un nivel de ruido de 0.0008.


/tmp/ipykernel_8072/3368324264.py:43: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [11]:
# ==============================================================================
# CELL: CAMPEÓN DEL TORNEO - DBSCAN
# ==============================================================================
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import geopandas as gpd
import plotly.express as px

display(Markdown("### 👑 **Contendiente #2: El Emperador DBSCAN**"))

# --- 1. PREPARACIÓN ---
# Usamos el mismo df_hdbscan_analysis de la celda anterior para consistencia
if 'df_hdbscan_analysis' not in locals():
    # Código de seguridad para recargar si es necesario
    query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
    df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)
    coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

# --- 2. PARÁMETROS Y EJECUCIÓN ---
# Probamos con un EPS un poco más grande para consolidar
EPSILON = 0.12
MIN_SAMPLES = 25

dbscan = DBSCAN(eps=EPSILON, min_samples=MIN_SAMPLES, n_jobs=-1)
cluster_labels = dbscan.fit_predict(coords_scaled)
df_hdbscan_analysis['dbscan_cluster_id'] = cluster_labels

# --- 3. ANÁLISIS ---
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados de DBSCAN:**\n*   **Clusters Consolidados:** `{n_clusters}`\n*   **Puntos de Ruido:** `{n_noise}`"))

# --- 4. VISUALIZACIÓN ---
fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="dropoff_lat",
    lon="dropoff_lon",
    color="dbscan_cluster_id",
    mapbox_style="carto-positron",
    zoom=10.5, height=800,
    title=f'Mapa de Clusters DBSCAN (eps={EPSILON}, k={MIN_SAMPLES})',
    color_continuous_scale=px.colors.qualitative.Plotly
)
fig.show()

### 👑 **Contendiente #2: El Emperador DBSCAN**

### 🗺️ **Resultados de DBSCAN:**
*   **Clusters Consolidados:** `5`
*   **Puntos de Ruido:** `1167`

/tmp/ipykernel_8072/1122874479.py:40: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [13]:
# ==============================================================================
# CELL: HDBSCAN CON JITTER Y PALETA DE ALTO CONTRASTE
# ==============================================================================
# !pip install hdbscan

import hdbscan
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**"))

# --- 1. RECONSTRUCCIÓN DE DATOS ---
query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)
coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
print(f"✅ Datos reconstruidos para HDBSCAN. Shape: {coords_scaled.shape}")

# --- 2. EJECUCIÓN DE HDBSCAN ---
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5)
print("⏳ HDBSCAN está pensando...")
cluster_labels = clusterer.fit_predict(coords_scaled)
df_hdbscan_analysis['hdbscan_cluster_id'] = cluster_labels
print("✅ Clustering HDBSCAN completado.")

# --- 3. ANÁLISIS DE RESULTADOS ---
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados:** {n_clusters} Clusters + {n_noise} Ruido"))

# --- 4. DOCTRINA "JITTER" ---
np.random.seed(42)
noise_level = 0.0008
df_hdbscan_analysis['lat_viz'] = df_hdbscan_analysis['dropoff_lat'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
df_hdbscan_analysis['lon_viz'] = df_hdbscan_analysis['dropoff_lon'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
print(f"✅ Doctrina 'Jitter' aplicada.")

# --- 5. VISUALIZACIÓN MEJORADA ---
# Convertir a string para que Plotly use una escala de colores discreta
df_hdbscan_analysis['cluster_id_str'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(str)

fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz",
    lon="lon_viz",
    color="cluster_id_str", # Usamos la columna string
    mapbox_style="carto-positron",
    zoom=10.5, height=900,
    title='Mapa de Clusters HDBSCAN (Alto Contraste)',
    # --- LA MAGIA DEL COLOR ---
    # Usamos una paleta CUALITATIVA (discreta) en lugar de continua
    color_discrete_sequence=px.colors.qualitative.Vivid,
    # Asignamos un color específico al ruido
    color_discrete_map={'-1': 'lightgrey'},
    category_orders={'cluster_id_str': sorted(df_hdbscan_analysis['cluster_id_str'].unique(), key=lambda x: int(x))}
)
fig.show()

### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**

✅ Datos reconstruidos para HDBSCAN. Shape: (4760, 2)
⏳ HDBSCAN está pensando...
✅ Clustering HDBSCAN completado.


### 🗺️ **Resultados:** 45 Clusters + 2144 Ruido

✅ Doctrina 'Jitter' aplicada.


/tmp/ipykernel_8072/1661412043.py:43: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [14]:
# ==============================================================================
# CELL: HDBSCAN CON JITTER Y PALETA DE ALTO CONTRASTE + tercer parametro
# ==============================================================================
# !pip install hdbscan

import hdbscan
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**"))

# --- 1. RECONSTRUCCIÓN DE DATOS ---
query_geo = "SELECT offer_id, dropoff_lat, dropoff_lon FROM offers WHERE dropoff_lat IS NOT NULL AND dropoff_lon IS NOT NULL"
df_hdbscan_analysis = pd.read_sql(query_geo, db_engine)
coords = df_hdbscan_analysis[['dropoff_lat', 'dropoff_lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)
print(f"✅ Datos reconstruidos para HDBSCAN. Shape: {coords_scaled.shape}")

# --- 2. EJECUCIÓN DE HDBSCAN ---
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_epsilon=0.00)
print("⏳ HDBSCAN está pensando...")
cluster_labels = clusterer.fit_predict(coords_scaled)
df_hdbscan_analysis['hdbscan_cluster_id'] = cluster_labels
print("✅ Clustering HDBSCAN completado.")

# --- 3. ANÁLISIS DE RESULTADOS ---
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
display(Markdown(f"### 🗺️ **Resultados:** {n_clusters} Clusters + {n_noise} Ruido"))

# --- 4. DOCTRINA "JITTER" ---
np.random.seed(42)
noise_level = 0.0008
df_hdbscan_analysis['lat_viz'] = df_hdbscan_analysis['dropoff_lat'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
df_hdbscan_analysis['lon_viz'] = df_hdbscan_analysis['dropoff_lon'] + np.random.normal(0, noise_level, len(df_hdbscan_analysis))
print(f"✅ Doctrina 'Jitter' aplicada.")

# --- 5. VISUALIZACIÓN MEJORADA ---
# Convertir a string para que Plotly use una escala de colores discreta
df_hdbscan_analysis['cluster_id_str'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(str)

fig = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz",
    lon="lon_viz",
    color="cluster_id_str", # Usamos la columna string
    mapbox_style="carto-positron",
    zoom=10.5, height=900,
    title='Mapa de Clusters HDBSCAN (Alto Contraste)',
    # --- LA MAGIA DEL COLOR ---
    # Usamos una paleta CUALITATIVA (discreta) en lugar de continua
    color_discrete_sequence=px.colors.qualitative.Vivid,
    # Asignamos un color específico al ruido
    color_discrete_map={'-1': 'lightgrey'},
    category_orders={'cluster_id_str': sorted(df_hdbscan_analysis['cluster_id_str'].unique(), key=lambda x: int(x))}
)
fig.show()

### 🤖 **Invocando a la Bestia: HDBSCAN con Jitter**

✅ Datos reconstruidos para HDBSCAN. Shape: (4760, 2)
⏳ HDBSCAN está pensando...
✅ Clustering HDBSCAN completado.


### 🗺️ **Resultados:** 45 Clusters + 2144 Ruido

✅ Doctrina 'Jitter' aplicada.


/tmp/ipykernel_8072/845281694.py:43: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [15]:
# ==============================================================================
# CELL: BAUTIZO DEFINITIVO (FINAL COMMIT)
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("### 🏛️ **Aplicando Nomenclatura Final (Bautizo Definitivo)**"))

# 1. DICCIONARIO MAESTRO OFICIAL
zone_naming_map = {
    -1: 'unassigned',
    0: 'viaducto_tlalpan',
    1: 'terminal_1_aicm',
    2: 'central_del_norte',
    3: 'terminal_2_aicm',       # Fusión
    4: 'terminal_2_aicm',       # Fusión
    5: 'plaza_satelite_mundo_e',
    6: 'lomas_verdes',
    7: 'pedregal',
    8: 'san_jeronimo_tizapan',
    9: 'san_angel_altavista',
    10: 'observatorio',
    11: 'sotelo_san_esteban',
    12: 'barranca_del_muerto',
    13: 'haciendas_san_fernando',
    14: 'vialidad_de_la_barranca',
    15: 'interlomas_magnocentro',
    16: 'santa_fe_itesm',
    17: 'cuajimalpa_centro',
    18: 'santa_fe_patio',
    19: 'santa_fe_abc',
    20: 'tamarindos',
    21: 'nodo_constituyentes_reforma',
    22: 'felix_cuevas',
    23: 'cruce_echanove',
    24: 'santa_fe_core',
    25: 'pabellon_bosques',
    26: 'duraznos',
    27: 'chamizal',
    28: 'lomas_anahuac',
    29: 'centro_historico',
    30: 'napoles_wtc',
    31: 'san_miguel_chapultepec',
    32: 'tacubaya',
    33: 'cofre_de_perote',
    34: 'condesa',
    35: 'roma_norte',
    36: 'col_juarez',
    37: 'la_diana',
    38: 'anzures_reforma',
    39: 'polanco_trebol',
    40: 'carso_antara_miyana',
    41: 'el_semaforo_de_palmas',
    42: 'campos_eliseos',
    43: 'virreyes',
    44: 'fc_cuernavaca'
}

# 2. APLICAR MAPEO
# Aseguramos que el ID sea entero para evitar errores de llave
df_hdbscan_analysis['hdbscan_cluster_id'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(int)

print(f"🔄 Bautizando {len(df_hdbscan_analysis)} registros...")
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['hdbscan_cluster_id'].map(zone_naming_map)

# 3. SAFETY CHECK (Limpieza de Huérfanos)
# Si quedó algún ID que no estaba en tu lista (ej. un cluster nuevo sorpresa), lo marcamos
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['zone_name'].fillna('ERROR_SIN_BAUTIZO')

# 4. CONFIRMACIÓN VISUAL
display(Markdown("#### **✅ Estado Final del Territorio**"))
conteo = df_hdbscan_analysis['zone_name'].value_counts().reset_index()
conteo.columns = ['Zona Bautizada', 'Total Ofertas']
display(conteo.head(15).style.background_gradient(cmap='Greens'))

# Verificación rápida de la fusión T2
t2_count = len(df_hdbscan_analysis[df_hdbscan_analysis['zone_name'] == 'terminal_2_aicm'])
print(f"\n✈️ Terminal 2 AICM consolidada con {t2_count} viajes.")

### 🏛️ **Aplicando Nomenclatura Final (Bautizo Definitivo)**

🔄 Bautizando 4760 registros...


#### **✅ Estado Final del Territorio**

,Zona Bautizada,Total Ofertas
0,unassigned,2144
1,santa_fe_core,331
2,carso_antara_miyana,170
3,terminal_1_aicm,111
4,terminal_2_aicm,111
5,sotelo_san_esteban,105
6,el_semaforo_de_palmas,96
7,vialidad_de_la_barranca,86
8,nodo_constituyentes_reforma,85
9,campos_eliseos,75



✈️ Terminal 2 AICM consolidada con 111 viajes.


In [16]:
# ==============================================================================
# CELL: VISUALIZACIÓN IN-NOTEBOOK (MAPA BAUTIZADO)
# ==============================================================================
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("### 🗺️ **Mapa Interactivo Opus: Zonas Bautizadas**"))

# 1. ACTUALIZAR NOMBRES (Para asegurar que lo que ves es la última versión)
if 'zone_naming_map' in locals():
    # Aseguramos tipos enteros para el mapeo
    df_hdbscan_analysis['hdbscan_cluster_id'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(int)
    # Mapeamos
    df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['hdbscan_cluster_id'].map(zone_naming_map)
    # Rellenamos huecos
    df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['zone_name'].fillna('SIN_NOMBRE')

# 2. ORDENAR PARA LEYENDA LIMPIA
df_hdbscan_analysis.sort_values(by='zone_name', inplace=True)

# 3. GENERAR EL MAPA
fig_inline = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz",      # Usamos las coordenadas visuales
    lon="lon_viz",

    # --- LA CLAVE: COLOREAR POR NOMBRE ---
    color="zone_name",
    # -------------------------------------

    hover_name="zone_name",
    hover_data={
        "hdbscan_cluster_id": True, # ID para referencia rápida
        "lat_viz": False,
        "lon_viz": False,
        "zone_name": False
    },
    mapbox_style="carto-positron",
    zoom=10.5,
    height=850, # Altura suficiente para ver bien en pantalla
    title='Territorios Opus Bautizados',
    color_discrete_sequence=px.colors.qualitative.Dark24
)

# 4. MOSTRAR EN PANTALLA
fig_inline.show()

### 🗺️ **Mapa Interactivo Opus: Zonas Bautizadas**

/tmp/ipykernel_8072/995764372.py:22: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_inline = px.scatter_mapbox(


# Auditar Bautizo de Zonas.. swapped SMCHvsTacubaya swapped InterlomasVsVialidadBarranca etc...

In [19]:
# ==============================================================================
# CELL: VOBO FINAL - LIMPIEZA Y VALIDACIÓN (IDS ESPECÍFICOS)
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

# 1. INPUT LIMPIO (Tu cadena cruda)
raw_input = "offer_idOF02371OF02849OF00133OF00159OF00717OF00826OF01075OF04235OF03264OF03827OF01097OF00286OF00297OF00300OF03053OF04721OF04459OF01494OF03140OF00441OF00879OF04276"

# 2. PARSEO INTELIGENTE
# Quitamos el encabezado accidental "offer_id" y extraemos bloques de 7 chars
clean_string = raw_input.replace("offer_id", "").strip()
target_ids = [clean_string[i:i+7] for i in range(0, len(clean_string), 7)]

display(Markdown(f"### 🧐 **Auditoría Final para {len(target_ids)} IDs Corregidos**"))
# print(f"IDs detectados: {target_ids}") # Descomenta si quieres ver la lista

# 3. OBTENER DIRECCIONES (CONTEXTO HUMANO)
# Usamos merge para garantizar que NO haya desfases
try:
    # Formato SQL
    ids_sql = "', '".join(target_ids)

    q_check = f"""
    SELECT offer_id, pickup_address, dropoff_address
    FROM offers
    WHERE offer_id IN ('{ids_sql}')
    """
    print("⏳ Consultando direcciones en base de datos...")
    df_addresses = pd.read_sql(q_check, db_engine)

except NameError:
    # Fallback si no hay conexión activa, busca en df_universe si existe
    print("⚠️ No hay conexión SQL activa. Buscando en memoria local...")
    if 'df_universe' in locals():
        df_addresses = df_universe[df_universe['offer_id'].isin(target_ids)][['offer_id', 'pickup_address', 'dropoff_address']]
    else:
        df_addresses = pd.DataFrame({'offer_id': target_ids, 'pickup_address': 'No DB', 'dropoff_address': 'No DB'})

# 4. OBTENER CLUSTERS (TU BAUTIZO)
# Traemos la info de tu análisis actual
cols_cluster = ['offer_id', 'hdbscan_cluster_id', 'zone_name', 'dropoff_lat', 'dropoff_lon']
# Filtramos solo columnas que existan para evitar errores
existing_cols = [c for c in cols_cluster if c in df_hdbscan_analysis.columns]

df_current_clusters = df_hdbscan_analysis[df_hdbscan_analysis['offer_id'].isin(target_ids)][existing_cols]

# 5. EL MERGE SEGURO (LA CLAVE DEL ÉXITO)
# Unimos usando 'offer_id' como llave maestra. Esto evita que se corran las filas.
df_final_vobo = pd.merge(df_addresses, df_current_clusters, on='offer_id', how='left')

# 6. VISUALIZACIÓN
# Reordenamos columnas para lectura lógica: ID -> De Dónde -> A Dónde -> Qué Zona Quedó
cols_order = ['offer_id', 'pickup_address', 'dropoff_address', 'zone_name', 'hdbscan_cluster_id']
final_cols = [c for c in cols_order if c in df_final_vobo.columns]

display(Markdown("#### **📋 Reporte de Validación (Sin Desfases)**"))
# Mostramos la tabla alineada a la izquierda
display(df_final_vobo[final_cols].style.set_properties(**{'text-align': 'left'}).background_gradient(subset=['hdbscan_cluster_id'], cmap='Pastel1'))

### 🧐 **Auditoría Final para 22 IDs Corregidos**

⏳ Consultando direcciones en base de datos...


#### **📋 Reporte de Validación (Sin Desfases)**

,offer_id,pickup_address,dropoff_address,zone_name,hdbscan_cluster_id
0,OF03264,"avenida paseo de la reforma, polanco / anzures","Av Juárez Esquina Eje Central S/N, Col Centro Histórico, 06050 Cuauhtémoc - Centro",centro_historico,29
1,OF03827,"paseo de la reforma, polanco / anzures","Av Juárez Esquina Eje Central S/N, Col Centro Histórico, 06050 Cuauhtémoc - Centro",centro_historico,29
2,OF00826,"calle felix berenguer, lomas de chapultepec","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
3,OF00159,"bosque de amates, bosques de las lomas","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
4,OF01075,"avenida vasco de quiroga, santa fe","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
5,OF04235,"vasco de quiroga loc d, santa fe","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
6,OF00717,"paseo de los tamarindos, bosques de las lomas","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
7,OF00133,"bosque de tabachines, bosques, ciudad de mexico, cdmx, mexico","Av Insurgentes Tacubaya 16A, Col Tacubaya Miguel Hidalgo, 11870 Ciudad de México, CMX - Condesa / Roma",san_miguel_chapultepec,31
8,OF03053,"paseo de los tamarindos, bosques","Av. Vasco de Quiroga 1340, La Mexicana, 01260 Álvaro Obregón, Ciudad de México - Camino Antiguo a Sante Fe",unassigned,-1
9,OF01097,"calzada chivatito, polanco / anzures","Av. Vasco de Quiroga #1401, Colonia Santa Fe Álvaro Obregón, 01219 Cuidad de México, Campo Militar 1F, Industrias Militares de Sedena, Álvaro Obregón, 01210 Ciudad de México, CDMX, México - Camino Antiguo a Sante Fe",unassigned,-1


In [ ]:
'''# ==============================================================================
# CELL: AUTO-BAUTIZO CONECTADO A GOOGLE DRIVE (SIN SUBIR ARCHIVOS)
# ==============================================================================
import pandas as pd
from google.colab import drive
from IPython.display import display, Markdown

# 1. MONTAR GOOGLE DRIVE
# Esto te pedirá autorización para leer tus archivos
print("🔌 Conectando a Google Drive...")
drive.mount('/content/drive')

# 2. DEFINIR LA RUTA (Traducción de Mac a Colab)
# Tu ruta local: /Volumes/GoogleDrive/My Drive/_Pienza/...
# Ruta en Colab: /content/drive/MyDrive/_Pienza/...
ruta_archivo = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/Pienza_Palette_Clusters_Full_Integrity.csv'

display(Markdown(f"### 🗳️ **Iniciando Protocolo de Recuperación desde Drive**"))
print(f"📂 Buscando archivo en: {ruta_archivo}")

try:
    # 3. CARGAR LA MEMORIA
    df_history = pd.read_csv(ruta_archivo)

    # Filtramos para quedarnos solo con nombres útiles
    df_valid_history = df_history[~df_history['zone_name'].isin(['unassigned', 'missing_coordinates', 'zona_desconocida'])]
    mapa_memoria = df_valid_history.set_index('offer_id')['zone_name'].to_dict()

    print(f"✅ Archivo leído exitosamente. {len(mapa_memoria)} etiquetas recuperadas.")

    # 4. APLICAR MEMORIA AL PRESENTE
    # Verificamos que exista el dataframe del HDBSCAN nuevo
    if 'df_hdbscan_analysis' not in locals():
        raise NameError("⚠️ Primero debes correr la celda de HDBSCAN para tener los nuevos clusters.")

    print("🔄 Cruzando datos históricos con los nuevos clusters...")
    df_hdbscan_analysis['nombre_historico'] = df_hdbscan_analysis['offer_id'].map(mapa_memoria)

    # 5. LA VOTACIÓN
    nuevo_mapa = {}
    nuevos_ids = sorted([x for x in df_hdbscan_analysis['hdbscan_cluster_id'].unique() if x != -1])

    print(f"📊 Re-calculando nombres para {len(nuevos_ids)} clusters...")

    for cluster_id in nuevos_ids:
        subset = df_hdbscan_analysis[df_hdbscan_analysis['hdbscan_cluster_id'] == cluster_id]
        conteo_votos = subset['nombre_historico'].value_counts()

        if not conteo_votos.empty:
            ganador = conteo_votos.index[0]
            nuevo_mapa[cluster_id] = ganador
        else:
            nuevo_mapa[cluster_id] = f"NUEVO_TERRITORIO_{cluster_id}"

    # Agregar ruido
    nuevo_mapa[-1] = 'Ruido / Periferia'

    # 6. IMPRIMIR RESULTADO
    print("\n" + "="*40)
    print("✨  TU NUEVO DICCIONARIO (COPIA Y PEGA ABAJO)  ✨")
    print("="*40)
    print("zone_naming_map = {")
    for k, v in sorted(nuevo_mapa.items()):
        print(f"    {k}: '{v}',")
    print("}")
    print("="*40)

except FileNotFoundError:
    print(f"🔴 ERROR: No encontré el archivo en esa ruta.")
    print(f"   Verifica que la carpeta '_Pienza' esté en la raíz de tu 'My Drive'.")
except Exception as e:
    print(f"⚠️ Error: {e}")'''

In [ ]:
'''# ==============================================================================
# CELL: COMMIT REVERT - MAPA SOLO CON IDs (PARA BAUTIZO MANUAL)
# ==============================================================================
import plotly.express as px
from google.colab import files

print("🧹 Generando mapa limpio (Solo IDs Numéricos)...")

# 1. LIMPIEZA DE VARIABLES (Para que no se crucen cables)
# Convertimos el ID a string para que Plotly use colores distintos (discretos), no un degradado.
# Ordenamos para que la leyenda salga bonita: -1, 0, 1, 2, 3...
df_hdbscan_analysis.sort_values(by='hdbscan_cluster_id', inplace=True)
df_hdbscan_analysis['id_visual'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(str)

# 2. GENERAR MAPA BASADO 100% EN EL ID
fig_manual = px.scatter_mapbox(
    df_hdbscan_analysis,
    lat="lat_viz",
    lon="lon_viz",

    # --- LA VERDAD DESNUDA ---
    color="id_visual",      # Colores por número
    hover_name="id_visual", # Etiqueta gigante con el número
    # -------------------------

    hover_data={
        "hdbscan_cluster_id": True,
        "id_visual": False,
        "lat_viz": False,
        "lon_viz": False
    },
    mapbox_style="carto-positron",
    zoom=10.5,
    height=900,
    title='Mapa de IDs Puros (Para Bautizo Manual)',
    # Usamos la paleta "Turbo" o "Jet" que tiene muchos colores para distinguir 45 zonas
    color_discrete_sequence=px.colors.sequential.Turbo
)

# 3. EXPORTAR
file_name = 'Opus_Mapa_Solo_IDs.html'
fig_manual.write_html(file_name)

print(f"✅ Mapa generado: {file_name}")
print("⬇️ Descargando...")
files.download(file_name)'''

In [ ]:
'''# ==============================================================================
# CELL: EXPORTACIÓN DE MAPA PARA VISUALIZACIÓN FULL-SCREEN
# ==============================================================================
from google.colab import files

# Nombre del archivo
file_name = 'Opus_HDBSCAN_Clusters_FullScreen_4.html'

print(f"💾 Guardando mapa interactivo como '{file_name}'...")

# Guardar la figura de Plotly (fig) que se creó en la celda anterior
try:
    fig.write_html(file_name)
    print("✅ Archivo HTML generado correctamente.")

    print("⬇️ Iniciando descarga automática...")
    files.download(file_name)
    print("🚀 ¡Listo! Abre el archivo descargado en tu navegador para verlo en pantalla gigante.")

except NameError:
    print("🔴 ERROR: No encuentro la variable 'fig'. Asegúrate de haber corrido la celda anterior (el mapa) primero.")
except Exception as e:
    print(f"⚠️ Error en la descarga: {e}")
    print(f"   -> Busca el archivo '{file_name}' en la carpeta de archivos de la izquierda (ícono de carpeta) y descárgalo manualmente.")
    '''

In [ ]:
'''# ==============================================================================
# CELL: BAUTIZO OFICIAL V2 - EL PROTOCOLO FINAL (PIENZA_PALETTE)
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("### 📜 **Aplicando Nomenclatura Oficial a Pienza_Palette**"))

# --- 1. EL DICCIONARIO MAESTRO (Traducción ID -> Nombre Operativo) ---
zone_naming_map = {
    -1: 'unassigned',
    0: 'viaducto_tlalpan',
    1: 'terminal_1_aicm',
    2: 'lindavista',
    3: 'terminal_2_aicm',
    4: 'plaza_satelite_mundo_e',
    5: 'lomas_verdes',
    6: 'pedregal',
    7: 'san_jeronimo_tizapan',
    8: 'san_angel_altavista',
    9: 'observatorio',
    10: 'lomas_de_sotelo',
    11: 'barranca_del_muerto',
    12: 'el_error_de_la_barranca',
    13: 'vialidad_de_la_barranca',
    14: 'interlomas_magnocentro',
    15: 'santa_fe_itesm',
    16: 'cuajimalpa_centro',
    17: 'santa_fe_patio',
    18: 'santa_fe_abc',
    19: 'nodo_constituyentes_reforma',
    20: 'tamarindos',
    21: 'felix_cuevas',
    22: 'cruce_echanove',
    23: 'duraznos',
    24: 'vistahermosa_colegios',
    25: 'chamizal',
    26: 'lomas_anahuac',
    27: 'santa_fe_centro_comercial',
    28: 'santa_fe_ogorman',
    29: 'santa_fe_barragan',
    30: 'santa_fe_samara',
    31: 'centro_historico',
    32: 'napoles_wtc',
    33: 'san_miguel_chapultepec',
    34: 'cofre_de_perote',
    35: 'condesa',
    36: 'roma_norte',
    37: 'col_juarez',
    38: 'la_diana',
    39: 'anzures_reforma',
    40: 'polanco_trebol',
    41: 'carso_antara_miyana',
    42: 'el_semaforo_de_palmas',
    43: 'virreyes',
    44: 'campos_eliseos'
}

# --- 2. APLICACIÓN DEL MAPEO ---
# Nos aseguramos de que el ID sea entero para coincidir con las llaves
try:
    df_hdbscan_analysis['hdbscan_cluster_id'] = df_hdbscan_analysis['hdbscan_cluster_id'].astype(int)
except:
    pass

# Mapeamos
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['hdbscan_cluster_id'].map(zone_naming_map)

# --- 3. GESTIÓN DE HUÉRFANOS (Safety Check) ---
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['zone_name'].fillna(
    'zona_desconocida_' + df_hdbscan_analysis['hdbscan_cluster_id'].astype(str)
)

print("✅ Bautizo completado exitosamente.")

# --- 4. VERIFICACIÓN FINAL (TOP ZONAS) ---
display(Markdown("#### **📊 Inventario Final de Territorios (Top 15 por Volumen)**"))
conteo_zonas = df_hdbscan_analysis['zone_name'].value_counts().reset_index()
conteo_zonas.columns = ['Zona Operativa', 'Total Ofertas']
display(conteo_zonas.head(15))'''

In [ ]:
'''# ==============================================================================
# CELL: BAUTIZO PURO 1:1 (SIN CONSOLIDACIÓN)
# ==============================================================================
from IPython.display import display, Markdown

# ASUNCIÓN: df_hdbscan_analysis con 'hdbscan_cluster_id' existe

display(Markdown("### 🏛️ **Registro Civil de Zonas: Bautizo Puro 1:1**"))

# 1. EL DICCIONARIO DEL ARQUITECTO (MAPEO 1:1)
zone_map = {
    -1: 'Ruido / Periferia',
    0: 'Estadio Azteca (W-Cup 2026)',
    1: 'AICM - Terminal 2',
    2: 'AICM - Terminal 1',
    3: 'AICM - Periferia',
    4: 'Fortuna / Lindavista',
    5: 'Lomas Verdes',
    6: 'Periferico (Atizapán/Tlalnepantla)',
    7: 'Pedregal',
    8: 'San Angel / Copilco',
    9: 'Naucalpan (San Esteban / 1 Mayo)',
    10: 'Observatorio',
    11: 'Mixcoac',
    12: 'Pepsico / Duraznos',
    13: 'Interlomas',
    14: 'Del Valle Sur',
    15: 'Santa Fe - Tec',
    16: 'Santa Fe - Zona ABC',
    17: 'Tamarindos / Saks Reforma',
    18: 'Cuajimalpa Centro',
    19: 'Universidad Anahuac',
    20: 'Servilletero (Loma de la Palma)',
    21: 'De Las Fuentes',
    22: 'El Yaqui / Cruce Echánove',
    23: 'Santa Fe - Patio',
    24: 'Santa Fe - Core',
    25: 'Santa Fe - Centro Comercial',
    26: 'Santa Fe - Corredor Ave. Santa Fe',
    27: 'Centro Histórico',
    28: 'Lomas de Sotelo',
    29: 'Nápoles WTC',
    30: 'Condesa / Roma Sur',
    31: 'Lomas - Fuentes / Cofre de Perote',
    32: 'Roma Norte',
    33: 'Lomas - San Isidro',
    34: 'Col. Juárez',
    35: 'Anzures / Cuahtemoc / Zona Rosa',
    36: 'Polanco - Antara/Carso',
    37: 'Polanco - Miyana',
    38: 'Lomas - Virreyes',
    39: 'Lomas - El Semáforo de Palmas',
    40: 'Polanco - Corredor FC Cuernavaca',
    41: 'Polanco - Anahuac (Lago Alberto)',
    42: 'Polanco - 5a Seccion',
    43: 'Polanco - Campos Eliseos (Hoteles)',
    44: 'Polanco - Campos Eliseos (Residencial)'
}

# 2. APLICAR EL BAUTIZO
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['hdbscan_cluster_id'].map(zone_map)
# Si algún ID no está en el mapa, se volverá NaN. Lo llenamos.
df_hdbscan_analysis['zone_name'] = df_hdbscan_analysis['zone_name'].fillna(df_hdbscan_analysis['hdbscan_cluster_id'].apply(lambda x: f'Cluster Sin Bautizar #{x}'))
print("✅ Clusters Bautizados.")

# 3. VERIFICAR LA DISTRIBUCIÓN
display(Markdown("#### **Frecuencia de las Zonas Bautizadas**"))
display(df_hdbscan_analysis['zone_name'].value_counts().reset_index())

# 4. VISUALIZACIÓN FINAL CONSOLIDADA
# Para que el ruido se vea bien, lo separamos
df_plot_clusters = df_hdbscan_analysis[df_hdbscan_analysis['zone_name'] != 'Ruido / Periferia']
df_plot_noise = df_hdbscan_analysis[df_hdbscan_analysis['zone_name'] == 'Ruido / Periferia']

fig = px.scatter_mapbox(
    df_plot_clusters,
    lat="lat_viz",
    lon="lon_viz",
    color="zone_name",
    mapbox_style="carto-positron",
    zoom=10.5, height=900,
    title='Mapa de Zonas Estratégicas (Bautizo 1:1)',
    hover_name='zone_name',
    # Usamos una paleta de alto contraste para los clusters
    color_discrete_sequence=px.colors.qualitative.Vivid
)

# --- LA CORRECCIÓN CLAVE ---
# Añadimos el ruido como una capa separada (trace) con estilo personalizado
fig.add_trace(
    go.Scattermapbox(
        lat=df_plot_noise['lat_viz'],
        lon=df_plot_noise['lon_viz'],
        mode='markers',
        marker=go.scattermapbox.Marker(
            size=5,
            color='lightgrey', # <-- El color que queríamos
            opacity=0.4
        ),
        name='Ruido / Periferia' # Nombre para la leyenda
    )
)

fig.update_layout(legend_title_text='Zona Estratégica')
fig.show()

In [ ]:
'''# ==============================================================================
# CELL: AUDITORÍA DE TEXTO DE CLUSTER (VERSIÓN AUTOCONTENida)
# ==============================================================================
from IPython.display import display, Markdown
import pandas as pd

display(Markdown("### 🕵️ **Iniciando Auditoría Forense de Cluster Geoespacial**"))

# --- FASE 1: RECONSTRUCCIÓN COMPLETA ---
# 1. Cargar el DataFrame MAESTRO con todas las columnas de texto
query_master = "SELECT offer_id, dropoff_address FROM offers"
df_master_audit = pd.read_sql(query_master, db_engine)
print(f"✅ DataFrame Maestro para auditoría cargado. Shape: {df_master_audit.shape}")

# 2. Cargar los resultados del Clustering (el df con los IDs de cluster)
# ASUNCIÓN: 'df_hdbscan_analysis' existe desde la celda de clustering.
if 'df_hdbscan_analysis' not in locals():
    print("🔴 ERROR CRÍTICO: df_hdbscan_analysis no existe. Ejecute la celda de clustering primero.")
else:
    # --- FASE 2: ENRIQUECIMIENTO ---
    # 3. Unir los resultados del clustering con los datos de texto
    df_audit_full = pd.merge(
        df_master_audit,
        df_hdbscan_analysis[['offer_id', 'zone_name']],
        on='offer_id',
        how='inner' # Usamos inner join para quedarnos solo con los puntos que fueron clusterizados
    )
    print("✅ Datos de cluster unidos con el DataFrame maestro.")

    # --- FASE 3: INTERROGATORIO ---
    # 4. NOMBRE DEL CLUSTER A AUDITAR
    target_zone = 'AICM - Terminal 2'

    # 5. FILTRAR Y MOSTRAR
    df_audit_target = df_audit_full[df_audit_full['zone_name'] == target_zone]
    num_points = len(df_audit_target)

    if num_points > 0:
        display(Markdown(f"\n### 📋 **Resultados para: `{target_zone}`** ({num_points} puntos)"))
        unique_addresses = df_audit_target['dropoff_address'].unique()

        print("Direcciones de destino únicas en este cluster:")
        for address in sorted(unique_addresses):
            print(f"- {address}")
    else:
        display(Markdown(f"\n⚠️ **Advertencia:** No se encontraron puntos para `{target_zone}`."))'''

In [ ]:
'''# ==============================================================================
# CELL: EXPORTACIÓN PARA AUDITORÍA GEOESPACIAL (A GOOGLE DRIVE)
# ==============================================================================
from IPython.display import display, Markdown

# ASUNCIÓN: df_hdbscan_analysis con 'zone_name' existe

if 'df_hdbscan_analysis' not in locals() or 'zone_name' not in df_hdbscan_analysis.columns:
    display(Markdown("🔴 **ERROR:** `df_hdbscan_analysis` no está definido. Ejecute las celdas de Bautizo primero."))
else:
    # --- RUTA DE DESTINO EN GOOGLE DRIVE ---
    output_path = '/content/drive/MyDrive/_Pienza/Assets/Phase_2/cluster_audit_iter3.csv'

    # 1. Seleccionar las columnas relevantes del DataFrame maestro
    query_full = "SELECT offer_id, pickup_address, dropoff_address, pickup_lat, pickup_lon, dropoff_lat, dropoff_lon FROM offers"
    df_export = pd.read_sql(query_full, db_engine)

    # 2. Unir con los resultados del clustering
    df_export = pd.merge(
        df_export,
        df_hdbscan_analysis[['offer_id', 'hdbscan_cluster_id', 'zone_name']],
        on='offer_id',
        how='left'
    )

    # 3. Guardar en Google Drive
    try:
        df_export.to_csv(output_path, index=False)
        print(f"✅ Archivo guardado exitosamente en Google Drive:")
        print(output_path)
    except Exception as e:
        print(f"🔴 ERROR al guardar en Google Drive: {e}")'''

In [ ]:
'''# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
# CELL A: LA FUSIÓN DE CORRECCIONES (VÍA PANDAS DIRECTO)
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
from IPython.display import display, Markdown
from google.colab import auth
import pandas as pd

display(Markdown("### 🩹 **Operación Pureza: Aplicando Hotfix Geoespacial**"))

# --- 1. CONSTRUIR EL URL DE LECTURA DIRECTA ---
# Ve a tu Google Sheet, haz clic en "Compartir" -> "Cualquier persona con el enlace".
# Copia el URL.
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1PTecbTRipi99sfAG6aS-GC-dwzl0vBXTWQHvhQ-7M-s/edit?usp=sharing'
SHEET_NAME = 'Sheet3'

# Transformamos el URL para la lectura directa de CSV
csv_export_url = SHEET_URL.replace('/edit?usp=sharing', f'/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}')

try:
    # --- 2. CARGAR CORRECCIONES DIRECTAMENTE CON PANDAS ---
    df_corrections = pd.read_csv(csv_export_url)
    df_corrections.columns = df_corrections.columns.str.lower()
    print(f"✅ {len(df_corrections)} correcciones cargadas directamente.")

    # --- 3. PREPARACIÓN DEL DATAFRAME MAESTRO ---
    query_master = "SELECT * FROM v_offers_human"
    df_master = pd.read_sql(query_master, db_engine)
    df_master.columns = df_master.columns.str.lower()

    # --- 4. LA CIRUGÍA EN MEMORIA ---
    df_master_indexed = df_master.set_index('offer_id')
    df_corrections_indexed = df_corrections.set_index('offer_id')

    df_corrections_indexed = df_corrections_indexed.rename(columns={'clean_lat': 'dropoff_lat', 'clean_lon': 'dropoff_lon'})
    df_master_indexed.update(df_corrections_indexed[['dropoff_lat', 'dropoff_lon']])
    df_master_patched = df_master_indexed.reset_index()

    print("✅ Coordenadas corruptas parcheadas en memoria.")
    print(f"   -> El nuevo DataFrame 'df_master_patched' está listo.")

except Exception as e:
    print(f"🔴 ERROR: No se pudo leer la hoja. ¿Está el URL correcto y compartido públicamente (lector)?")
    print(f"   -> Detalle: {e}")
    '''

In [ ]:
'''# ==============================================================================
# CELL B: HDBSCAN SOBRE DATOS PARCHEADOS (COMPLETA)
# ==============================================================================
import hdbscan
import plotly.express as px
from IPython.display import display, Markdown
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import plotly.graph_objects as go

# ASUNCIÓN: `df_master_patched` existe desde la Celda A

if 'df_master_patched' not in locals():
    display(Markdown("🔴 **ERROR CRÍTICO:** El DataFrame `df_master_patched` no existe. Ejecute la Celda A primero."))
else:
    display(Markdown("### 🤖 **Invocando a la Bestia HDBSCAN (sobre Datos Puros)**"))

    # --- 1. PREPARACIÓN DE DATOS ---
    # Usamos una copia para seguridad
    df_geo = df_master_patched[['offer_id', 'dropoff_lat', 'dropoff_lon']].copy()
    # Eliminar cualquier fila que todavía pueda tener nulos en coordenadas
    df_geo.dropna(subset=['dropoff_lat', 'dropoff_lon'], inplace=True)

    coords = df_geo[['dropoff_lat', 'dropoff_lon']]
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)
    print(f"✅ Datos geoespaciales purificados listos para HDBSCAN. Shape: {coords_scaled.shape}")

    # --- 2. EJECUCIÓN DE HDBSCAN ---
    # Parámetros probados
    clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5)

    print("⏳ HDBSCAN está pensando...")
    cluster_labels = clusterer.fit_predict(coords_scaled)

    # Asignar los labels al DataFrame geo
    df_geo['hdbscan_cluster_id'] = cluster_labels
    print("✅ Clustering HDBSCAN completado.")

    # --- 3. ANÁLISIS DE RESULTADOS ---
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    display(Markdown(f"### 🗺️ **Resultados:** {n_clusters} Clusters + {n_noise} Ruido"))

    # --- 4. DOCTRINA "JITTER" PARA VISUALIZACIÓN ---
    np.random.seed(42)
    noise_level = 0.0008
    df_geo['lat_viz'] = df_geo['dropoff_lat'] + np.random.normal(0, noise_level, len(df_geo))
    df_geo['lon_viz'] = df_geo['dropoff_lon'] + np.random.normal(0, noise_level, len(df_geo))

    # --- 5. VISUALIZACIÓN HÍBRIDA (LA MEJOR VERSIÓN) ---
    df_clusters = df_geo[df_geo['hdbscan_cluster_id'] != -1].copy()
    df_noise = df_geo[df_geo['hdbscan_cluster_id'] == -1].copy()

    df_clusters['cluster_id_str'] = df_clusters['hdbscan_cluster_id'].astype(str)

    # Crear figura base con los clusters
    fig = px.scatter_mapbox(
        df_clusters,
        lat="lat_viz",
        lon="lon_viz",
        color="cluster_id_str",
        mapbox_style="carto-positron",
        zoom=10.5, height=900,
        title='Mapa de Clusters HDBSCAN (Datos Puros)',
        color_discrete_sequence=px.colors.qualitative.Vivid
    )

    # Añadir capa de ruido
    fig.add_trace(go.Scattermapbox(
        lat=df_noise['lat_viz'],
        lon=df_noise['lon_viz'],
        mode='markers',
        marker=go.scattermapbox.Marker(size=5, color='lightgrey', opacity=0.3),
        name='Ruido'
    ))

    fig.update_layout(legend_title_text='Zone ID')
    fig.show()

In [ ]:
'''# ==============================================================================
# CELL: MAPA DE ADOPCIÓN RÁPIDA (HDBSCAN + KNN)
# ==============================================================================
from sklearn.neighbors import KNeighborsClassifier
import plotly.express as px
from IPython.display import display, Markdown

# ASUNCIÓN: df_hdbscan_analysis, coords_scaled y cluster_labels existen de la celda anterior.

if 'df_hdbscan_analysis' not in locals():
    print("🔴 ERROR: Ejecute la celda de HDBSCAN primero.")
else:
    display(Markdown("### 🤝 **Protocolo de Adopción de Huérfanos (KNN)**"))

    # 1. Separar datos
    known_data = coords_scaled[cluster_labels != -1]
    known_labels = cluster_labels[cluster_labels != -1]
    noise_data = coords_scaled[cluster_labels == -1]

    if len(noise_data) > 0 and len(known_data) > 0:
        # 2. Entrenar KNN
        print("⏳ Entrenando KNN para re-asignar el ruido...")
        knn = KNeighborsClassifier(n_neighbors=1)
        knn.fit(known_data, known_labels)

        # 3. Predecir y Re-asignar
        noise_predictions = knn.predict(noise_data)
        final_labels = cluster_labels.copy()
        final_labels[cluster_labels == -1] = noise_predictions
        df_hdbscan_analysis['knn_cluster_id'] = final_labels
        print("✅ Puntos de ruido re-asignados.")

        # 4. Visualizar
        df_hdbscan_analysis['knn_cluster_id_str'] = df_hdbscan_analysis['knn_cluster_id'].astype(str)

        fig = px.scatter_mapbox(
            df_hdbscan_analysis,
            lat="lat_viz",
            lon="lon_viz",
            color="knn_cluster_id_str",
            mapbox_style="carto-positron",
            zoom=10.5, height=900,
            title='Mapa de Zonas Consolidadas por KNN',
            color_discrete_sequence=px.colors.qualitative.Vivid
        )
        fig.show()
    else:
        print("ℹ️ No hay puntos de ruido que re-asignar o no hay clusters base.")